In [1]:
pip install pymupdf pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.5/22.5 MB 24.7 MB/s  0:00:004.7 MB/s eta 0:00:01:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import re
import pandas as pd
import fitz  # PyMuPDF


# ----------------------------
# 1) USER SETTINGS (EDIT THESE)
# ----------------------------

BASE = Path(".").resolve()

PDF_PATH = BASE / "data_intermediate" / "pdfs" / "sbm_report_2024.pdf"

TICKER = "sbm"
YEAR = 2024

PAGE_START = 5
PAGE_END   = 35

EXCLUDE_PAGES = []

OUT_TEXT_FOLDER = BASE / "data_intermediate" / f"mdna_text_{YEAR}"
OUT_LOG_FOLDER  = BASE / "outputs" / "logs"

OUT_TEXT_FOLDER.mkdir(parents=True, exist_ok=True)
OUT_LOG_FOLDER.mkdir(parents=True, exist_ok=True)


# -----------------------------------
# 2) FILTERS: remove tables etc
# -----------------------------------

def looks_like_table_line(line: str) -> bool:
    s = line.strip()
    if not s:
        return True

    digits = sum(ch.isdigit() for ch in s)
    letters = sum(ch.isalpha() for ch in s)
    length = len(s)
    digit_ratio = digits / max(1, length)

    num_tokens = re.findall(r"\b\d+([.,]\d+)?\b", s)
    num_token_count = len(num_tokens)

    if num_token_count >= 3:
        return True
    if digit_ratio > 0.30 and letters < 30:
        return True
    if re.search(r"\b(eur|€|%|kpi|kpis|scope\s*[123])\b", s.lower()) and num_token_count >= 2:
        return True

    return False


def clean_and_filter_lines(text_block: str) -> str:
    lines = [ln.rstrip() for ln in text_block.splitlines()]
    kept = []

    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        if looks_like_table_line(s):
            continue
        kept.append(s)

    return "\n".join(kept)


def normalize_text(text: str) -> str:
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)     # dehyphenate
    text = re.sub(r"\n{3,}", "\n\n", text)          # collapse blank lines
    text = "\n".join([ln.rstrip() for ln in text.splitlines()])
    return text.strip()


# ----------------------------
# 3) PDF EXTRACTION (PyMuPDF)
# ----------------------------

def extract_mdna_from_pdf(pdf_path: Path, page_start: int, page_end: int, exclude_pages=None) -> dict:
    exclude_pages = set(exclude_pages or [])
    doc = fitz.open(pdf_path)

    if page_start < 0 or page_end >= doc.page_count or page_start > page_end:
        doc.close()
        raise ValueError(f"Invalid page range. PDF pages: 0..{doc.page_count-1}")

    extracted_pages = []
    flagged_pages_low_text = []
    page_stats = []

    for pno in range(page_start, page_end + 1):

        if pno in exclude_pages:
            page_stats.append({
                "pdf_page_index": pno,
                "raw_text_len": 0,
                "kept_text_len": 0,
                "kept_blocks": 0,
                "status": "EXCLUDED_PAGE"
            })
            continue

        page = doc.load_page(pno)
        blocks = page.get_text("blocks")  # includes text + non-text blocks

        kept_blocks = []
        raw_text_len = 0

        for b in blocks:
            # (x0, y0, x1, y1, "text", block_no, block_type)
            block_text = b[4]
            block_type = b[6] if len(b) >= 7 else 0

            if not block_text or not block_text.strip():
                continue

            raw_text_len += len(block_text)

            # only keep TEXT blocks
            if block_type != 0:
                continue

            filtered = clean_and_filter_lines(block_text)
            if filtered.strip():
                kept_blocks.append(filtered)

        page_text = "\n\n".join(kept_blocks).strip()

        page_stats.append({
            "pdf_page_index": pno,
            "raw_text_len": raw_text_len,
            "kept_text_len": len(page_text),
            "kept_blocks": len(kept_blocks),
            "status": "OK"
        })

        if len(page_text) < 200:
            flagged_pages_low_text.append(pno)

        extracted_pages.append(page_text)

    doc.close()

    full_text = "\n\n".join([t for t in extracted_pages if t]).strip()
    full_text = normalize_text(full_text)

    return {
        "text": full_text,
        "page_stats": pd.DataFrame(page_stats),
        "flagged_pages_low_text": flagged_pages_low_text
    }


# ----------------------------
# 4) RUN + SAVE + LOG
# ----------------------------

result = extract_mdna_from_pdf(PDF_PATH, PAGE_START, PAGE_END, exclude_pages=EXCLUDE_PAGES)
mdna_text = result["text"]
page_stats = result["page_stats"]
flagged = result["flagged_pages_low_text"]

out_txt = OUT_TEXT_FOLDER / f"{TICKER}_{YEAR}.txt"
out_stats = OUT_TEXT_FOLDER / f"{TICKER}_{YEAR}_page_stats.csv"

out_txt.write_text(mdna_text, encoding="utf-8")
page_stats.to_csv(out_stats, index=False)

word_count = len(mdna_text.split())

print("Saved MD&A text to:", out_txt)
print("Saved page-level stats to:", out_stats)
print("Total extracted characters:", len(mdna_text))
print("Total extracted words:", word_count)

if flagged:
    print("\nWARNING: pages with low extracted text (possible scanned pages or aggressive filtering):")
    print(flagged)

print("\n--- FIRST 600 CHARS ---\n", mdna_text[:600])
print("\n--- LAST 600 CHARS ---\n", mdna_text[-600:])

log_path = OUT_LOG_FOLDER / "mdna_extraction_log.csv"
log_row = pd.DataFrame([{
    "ticker": TICKER,
    "year": YEAR,
    "pdf_path": str(PDF_PATH),
    "page_start_pdf_index": PAGE_START,
    "page_end_pdf_index": PAGE_END,
    "excluded_pages_pdf_index": ",".join(map(str, EXCLUDE_PAGES)) if EXCLUDE_PAGES else "",
    "extracted_chars": len(mdna_text),
    "extracted_words": word_count,
    "flagged_pages_low_text": ",".join(map(str, flagged)) if flagged else ""
}])

if log_path.exists():
    existing = pd.read_csv(log_path)
    combined = pd.concat([existing, log_row], ignore_index=True)
else:
    combined = log_row

combined.to_csv(log_path, index=False)
print("\nUpdated extraction log:", log_path)


FileNotFoundError: no such file: '/Users/pnlmf036/Documents/Thesis_Textanalysis/data_intermediate/pdfs/sbm_report_2024.pdf'